In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm
import cv2

def preprocess_images(input_path, output_path, target_size=(224,224)):
    # Process each subdirectory separately
    for root, dirs, files in os.walk(input_path):
        # Get the relative path from input_path
        relative_path = os.path.relpath(root, input_path)
        # Create corresponding output directory
        current_output_dir = os.path.join(output_path, relative_path)
        os.makedirs(current_output_dir, exist_ok=True)

        # Process images in current directory
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                try:
                    # Full path for input and output images
                    input_img_path = os.path.join(root, file)
                    output_img_path = os.path.join(current_output_dir,
                                                 os.path.splitext(file)[0] + '.jpg')

                    # Read and process image
                    img = cv2.imread(input_img_path)
                    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    resized = cv2.resize(gray, target_size, interpolation=cv2.INTER_AREA)

                    # Apply contrast enhancement
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    enhanced = clahe.apply(resized)

                    # Denoise image
                    denoised = cv2.fastNlMeansDenoising(enhanced)

                    # Apply median blur
                    filtered = cv2.medianBlur(denoised, 3)

                    # Adjust contrast and brightness
                    alpha = 1.2
                    beta = 10
                    adjusted = cv2.convertScaleAbs(filtered, alpha=alpha, beta=beta)

                    # Save processed image
                    cv2.imwrite(output_img_path, adjusted, [cv2.IMWRITE_JPEG_QUALITY, 95])

                except Exception as e:
                    print(f"Error processing {input_img_path}: {str(e)}")

# Define paths
input_path = 'C:\\Users\\sa\\Desktop\\Training'
output_path = 'C:\\Users\\sa\\Desktop\\Training2'

# Run the preprocessing
preprocess_images(input_path, output_path)

# Verify results
def verify_dataset(path):
    image_details = {}
    for root, _, files in os.walk(path):
        folder_name = os.path.basename(root)
        image_details[folder_name] = len([f for f in files if f.lower().endswith('.jpg')])
    return image_details

# Print verification results
processed_images = verify_dataset(output_path)
print("\nProcessing Results:")
print("Images in each folder:")
for folder, count in processed_images.items():
    if folder != '': # Skip empty folder names
        print(f"{folder}: {count} images")



Processing Results:
Images in each folder:
Training2: 0 images
glioma: 1329 images
meningioma: 1339 images
notumor: 1595 images
pituitary: 1481 images


In [ ]:
!pip install imbalanced-learn scikit-learn opencv-python numpy tqdm


In [ ]:
import os
import numpy as np
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.preprocessing import LabelEncoder
import cv2
from tqdm import tqdm

def load_dataset(data_path):
    images = []
    labels = []

    for class_folder in os.listdir(data_path):
        folder_path = os.path.join(data_path, class_folder)
        if os.path.isdir(folder_path):
            print(f"Loading {class_folder}")
            for img_name in tqdm(os.listdir(folder_path)):
                if img_name.lower().endswith('.jpg'):
                    img_path = os.path.join(folder_path, img_name)
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    img_flat = img.flatten()
                    images.append(img_flat)
                    labels.append(class_folder)

    return np.array(images), np.array(labels)

# Path to your preprocessed dataset
data_path = "C:\\Users\\sa\\Desktop\\Training2"

# Load the dataset
X, y = load_dataset(data_path)

# Convert labels to numerical format
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Apply BorderlineSMOTE with optimal parameters for brain tumor images
borderline_smote = BorderlineSMOTE(
    random_state=42,
    k_neighbors=5,
    m_neighbors=10,
    kind='borderline-1'
)
X_balanced, y_balanced = borderline_smote.fit_resample(X, y_encoded)

def validate_synthetic_image(image, min_variance=100, min_range=50):
    if np.var(image) < min_variance:
        return False
    if np.max(image) - np.min(image) < min_range:
        return False
    return True

def save_balanced_images(X_balanced, y_balanced, output_path):
    os.makedirs(output_path, exist_ok=True)

    # Create class folders
    class_names = le.inverse_transform(np.unique(y_balanced))
    for class_name in class_names:
        os.makedirs(os.path.join(output_path, class_name), exist_ok=True)

    # Save images with quality validation
    for idx, (img_flat, label) in enumerate(zip(X_balanced, y_balanced)):
        img = img_flat.reshape(224, 224)
        if validate_synthetic_image(img):
            class_name = le.inverse_transform([label])[0]
            output_file = os.path.join(output_path, class_name, f'balanced_img_{idx}.jpg')
            cv2.imwrite(output_file, img)

# Save balanced dataset
output_path = 'C:\\Users\\sa\\Desktop\\Balanced_Training1'
save_balanced_images(X_balanced, y_balanced, output_path)

# Print balancing results
print("\nClass Distribution Before SMOTE:")
for class_name in le.classes_:
    count = np.sum(y == class_name)
    print(f"{class_name}: {count}")

print("\nClass Distribution After SMOTE:")
for class_name, label in zip(le.classes_, range(len(le.classes_))):
    count = np.sum(y_balanced == label)
    print(f"{class_name}: {count}")


Loading glioma


100%|██████████| 1329/1329 [00:00<00:00, 4809.47it/s]


Loading meningioma


100%|██████████| 1339/1339 [00:00<00:00, 4624.53it/s]


Loading notumor


100%|██████████| 1595/1595 [00:00<00:00, 4063.42it/s]


Loading pituitary


100%|██████████| 1481/1481 [00:00<00:00, 4206.51it/s]
C:\Users\sa\.conda\envs\fiza\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\sa\.conda\envs\fiza\lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The BorderlineSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(



Class Distribution Before SMOTE:
glioma: 1329
meningioma: 1339
notumor: 1595
pituitary: 1481

Class Distribution After SMOTE:
glioma: 1595
meningioma: 1595
notumor: 1595
pituitary: 1595


In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm
import cv2

def preprocess_images(input_path, output_path, target_size=(224,224)):
    # Process each subdirectory separately
    for root, dirs, files in os.walk(input_path):
        # Get the relative path from input_path
        relative_path = os.path.relpath(root, input_path)
        # Create corresponding output directory
        current_output_dir = os.path.join(output_path, relative_path)
        os.makedirs(current_output_dir, exist_ok=True)

        # Process images in current directory
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                try:
                    # Full path for input and output images
                    input_img_path = os.path.join(root, file)
                    output_img_path = os.path.join(current_output_dir,
                                                 os.path.splitext(file)[0] + '.jpg')

                    # Read and process image
                    img = cv2.imread(input_img_path)
                    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    resized = cv2.resize(gray, target_size, interpolation=cv2.INTER_AREA)

                    # Apply contrast enhancement
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    enhanced = clahe.apply(resized)

                    # Denoise image
                    denoised = cv2.fastNlMeansDenoising(enhanced)

                    # Apply median blur
                    filtered = cv2.medianBlur(denoised, 3)

                    # Adjust contrast and brightness
                    alpha = 1.2
                    beta = 10
                    adjusted = cv2.convertScaleAbs(filtered, alpha=alpha, beta=beta)

                    # Save processed image
                    cv2.imwrite(output_img_path, adjusted, [cv2.IMWRITE_JPEG_QUALITY, 95])

                except Exception as e:
                    print(f"Error processing {input_img_path}: {str(e)}")

# Define paths
input_path = "C:\\Users\\sa\\Desktop\\Testing"
output_path = "C:\\Users\\sa\\Desktop\\Testing2"

# Run the preprocessing
preprocess_images(input_path, output_path)

# Verify results
def verify_dataset(path):
    image_details = {}
    for root, _, files in os.walk(path):
        folder_name = os.path.basename(root)
        image_details[folder_name] = len([f for f in files if f.lower().endswith('.jpg')])
    return image_details

# Print verification results
processed_images = verify_dataset(output_path)
print("\nProcessing Results:")
print("Images in each folder:")
for folder, count in processed_images.items():
    if folder != '': # Skip empty folder names
        print(f"{folder}: {count} images")



Processing Results:
Images in each folder:
Testing2: 0 images
glioma: 300 images
meningioma: 306 images
notumor: 405 images
pituitary: 300 images


In [ ]:
import os
import numpy as np
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.preprocessing import LabelEncoder
import cv2
from tqdm import tqdm

def load_dataset(data_path):
    images = []
    labels = []

    for class_folder in os.listdir(data_path):
        folder_path = os.path.join(data_path, class_folder)
        if os.path.isdir(folder_path):
            print(f"Loading {class_folder}")
            for img_name in tqdm(os.listdir(folder_path)):
                if img_name.lower().endswith('.jpg'):
                    img_path = os.path.join(folder_path, img_name)
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    img_flat = img.flatten()
                    images.append(img_flat)
                    labels.append(class_folder)

    return np.array(images), np.array(labels)

# Path to your preprocessed dataset
data_path =  "C:\\Users\\sa\\Desktop\\Testing2"

# Load the dataset
X, y = load_dataset(data_path)

# Convert labels to numerical format
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Apply BorderlineSMOTE with optimal parameters for brain tumor images
borderline_smote = BorderlineSMOTE(
    random_state=42,
    k_neighbors=5,
    m_neighbors=10,
    kind='borderline-1'
)
X_balanced, y_balanced = borderline_smote.fit_resample(X, y_encoded)

def validate_synthetic_image(image, min_variance=100, min_range=50):
    if np.var(image) < min_variance:
        return False
    if np.max(image) - np.min(image) < min_range:
        return False
    return True

def save_balanced_images(X_balanced, y_balanced, output_path):
    os.makedirs(output_path, exist_ok=True)

    # Create class folders
    class_names = le.inverse_transform(np.unique(y_balanced))
    for class_name in class_names:
        os.makedirs(os.path.join(output_path, class_name), exist_ok=True)

    # Save images with quality validation
    for idx, (img_flat, label) in enumerate(zip(X_balanced, y_balanced)):
        img = img_flat.reshape(224, 224)
        if validate_synthetic_image(img):
            class_name = le.inverse_transform([label])[0]
            output_file = os.path.join(output_path, class_name, f'balanced_img_{idx}.jpg')
            cv2.imwrite(output_file, img)

# Save balanced dataset
output_path = 'C:\\Users\\sa\\Desktop\\Balanced_testing2'
save_balanced_images(X_balanced, y_balanced, output_path)

# Print balancing results
print("\nClass Distribution Before SMOTE:")
for class_name in le.classes_:
    count = np.sum(y == class_name)
    print(f"{class_name}: {count}")

print("\nClass Distribution After SMOTE:")
for class_name, label in zip(le.classes_, range(len(le.classes_))):
    count = np.sum(y_balanced == label)
    print(f"{class_name}: {count}")


Loading glioma


100%|██████████| 300/300 [00:00<00:00, 4726.05it/s]


Loading meningioma


100%|██████████| 306/306 [00:00<00:00, 4528.83it/s]


Loading notumor


100%|██████████| 405/405 [00:00<00:00, 4440.56it/s]


Loading pituitary


100%|██████████| 300/300 [00:00<00:00, 3866.18it/s]
C:\Users\sa\.conda\envs\fiza\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\sa\.conda\envs\fiza\lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The BorderlineSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(



Class Distribution Before SMOTE:
glioma: 300
meningioma: 306
notumor: 405
pituitary: 300

Class Distribution After SMOTE:
glioma: 405
meningioma: 405
notumor: 405
pituitary: 405
